# Project 87 — Local Screenshot Debugging Agent
## Error Screenshot → Diagnosis → Fix Suggestion

**Stack:** LangChain · Ollama · Pydantic · Jupyter

*Uses text-based error simulation for local-first approach.*

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## Step 1 — Simulated Error Screenshots (as text)

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json

llm = ChatOllama(model="qwen3:8b", temperature=0.1)

error_screens = [
    {
        "id": "err_001",
        "context": "Python web application",
        "error_text": """Traceback (most recent call last):
  File "app.py", line 42, in handle_request
    result = db.query(User).filter_by(email=email).first()
  File "sqlalchemy/orm/query.py", line 3425, in first
    return self._iter().first()
sqlalchemy.exc.OperationalError: (psycopg2.OperationalError)
could not connect to server: Connection refused
    Is the server running on host "localhost" and accepting
    TCP/IP connections on port 5432?"""
    },
    {
        "id": "err_002",
        "context": "React frontend build",
        "error_text": """ERROR in ./src/components/Dashboard.tsx
Module not found: Error: Can't resolve '@/utils/formatDate' in
'/app/src/components'
  @ ./src/components/Dashboard.tsx 3:0-45
  @ ./src/App.tsx
  @ ./src/index.tsx

webpack compiled with 1 error"""
    },
    {
        "id": "err_003",
        "context": "Docker deployment",
        "error_text": """Error response from daemon: driver failed programming external
connectivity on endpoint api-server: Bind for 0.0.0.0:8080 failed:
port is already allocated.

docker: Error response from daemon: OCI runtime create failed:
container_linux.go:349: starting container process caused:
exec: "python": executable file not found in $PATH"""
    },
    {
        "id": "err_004",
        "context": "Kubernetes pod crash",
        "error_text": """NAME          READY   STATUS             RESTARTS   AGE
api-server    0/1     CrashLoopBackOff   5          10m

Events:
  Warning  BackOff  2m  kubelet  Back-off restarting failed container
  Warning  Failed   3m  kubelet  Error: secret "db-credentials" not found"""
    },
]
print(f"Error screens to debug: {len(error_screens)}")

## Step 2 — Automated Diagnosis

In [ ]:
class Diagnosis(BaseModel):
    error_type: str
    root_cause: str
    severity: str = Field(description="critical, high, medium, low")
    affected_component: str
    fix_steps: list[str]
    verification_command: str
    prevention_tip: str

debugger = llm.with_structured_output(Diagnosis)

diagnoses = []
for screen in error_screens:
    diag = debugger.invoke(
        f"Debug this error from a {screen['context']}:\n\n{screen['error_text']}"
    )
    diagnoses.append({"id": screen["id"], "context": screen["context"], "diagnosis": diag})
    print(f"\n{screen['id']} ({screen['context']}):")
    print(f"  Type: {diag.error_type}")
    print(f"  Root cause: {diag.root_cause}")
    print(f"  Severity: {diag.severity}")
    print(f"  Fix steps: {len(diag.fix_steps)}")

## Step 3 — Generate Fix Scripts

In [ ]:
fix_prompt = ChatPromptTemplate.from_messages([
    ("system", "Generate a bash script that fixes this error. Include comments. "
     "Make it safe to run (check before acting)."),
    ("human", "Error: {error}\nRoot cause: {cause}\nFix steps: {steps}\n\nBash fix script:")
])
fix_chain = fix_prompt | llm | StrOutputParser()

for d in diagnoses[:3]:
    diag = d["diagnosis"]
    fix_script = fix_chain.invoke({
        "error": diag.error_type,
        "cause": diag.root_cause,
        "steps": "\n".join(f"- {s}" for s in diag.fix_steps),
    })
    print(f"\n--- Fix for {d['id']} ---")
    print(fix_script[:300])
    print("...")

## Step 4 — Knowledge Base Builder

In [ ]:
# Build a reusable debugging knowledge base
print("DEBUGGING KNOWLEDGE BASE")
print("=" * 50)
kb_entries = []
for d in diagnoses:
    diag = d["diagnosis"]
    entry = {
        "error_type": diag.error_type,
        "component": diag.affected_component,
        "severity": diag.severity,
        "root_cause": diag.root_cause,
        "fix": diag.fix_steps[0] if diag.fix_steps else "N/A",
        "prevention": diag.prevention_tip,
        "verify": diag.verification_command,
    }
    kb_entries.append(entry)
    print(f"\n  [{diag.severity.upper()}] {diag.error_type}")
    print(f"  Cause: {diag.root_cause}")
    print(f"  Fix: {diag.fix_steps[0] if diag.fix_steps else 'N/A'}")
    print(f"  Prevent: {diag.prevention_tip}")

from pathlib import Path
Path("sample_data").mkdir(exist_ok=True)
with open("sample_data/debug_knowledge_base.json", "w") as f:
    json.dump(kb_entries, f, indent=2)
print(f"\n✓ Saved {len(kb_entries)} entries to knowledge base")

## What You Learned
- **Error pattern recognition** from screenshots/logs
- **Automated root cause analysis** with severity
- **Fix script generation** with safety checks
- **Knowledge base building** from resolved issues